[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/573phn/dh_ad_a3/blob/main/a3.ipynb)

# Installing required libraries

In [1]:
!pip install ollama

In [2]:
!sudo apt update && sudo apt install pciutils lshw

Hit:1 https://cli.github.com/packages stable InRelease
Get:2 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:3 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease [1,581 B]
Get:4 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Get:5 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Get:6 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  Packages [2,480 kB]
Hit:7 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:8 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:9 https://r2u.stat.illinois.edu/ubuntu jammy/main amd64 Packages [2,955 kB]
Hit:10 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:11 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu jammy InRelease
Get:12 http://security.ubuntu.com/ubuntu jammy-security/universe amd64 Packages [1,310 kB]
Hit:13 https://ppa.launchpadcontent.net/ubuntugis/ppa

In [3]:
!apt-get install -y zstd

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following NEW packages will be installed:
  zstd
0 upgraded, 1 newly installed, 0 to remove and 53 not upgraded.
Need to get 603 kB of archives.
After this operation, 1,695 kB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu jammy/main amd64 zstd amd64 1.4.8+dfsg-3build1 [603 kB]
Fetched 603 kB in 1s (556 kB/s)
Selecting previously unselected package zstd.
(Reading database ... 122387 files and directories currently installed.)
Preparing to unpack .../zstd_1.4.8+dfsg-3build1_amd64.deb ...
Unpacking zstd (1.4.8+dfsg-3build1) ...
Setting up zstd (1.4.8+dfsg-3build1) ...
Processing triggers for man-db (2.10.2-1) ...


### Installing Ollama

In [4]:
!curl -fsSL https://ollama.com/install.sh | sh

>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%
>>> Creating ollama user...
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> NVIDIA GPU installed.
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.


### Starting Ollama server and saving its output to a log file ollama.log

In [5]:
!nohup ollama serve > ollama.log 2>&1 &

### Download the gemma3 model

In [12]:
!ollama pull gemma3

### List downloaded models

In [13]:
!ollama list

NAME             ID              SIZE      MODIFIED       
gemma3:latest    a2af6cc3eb7f    3.3 GB    57 seconds ago    


# Assignment 3 - Part 2

In [14]:
import ollama
import pandas as pd
from sklearn.metrics import classification_report

In [15]:
def prompt_ollama(lyrics: str, train_df: pd.DataFrame, shots: int = 0):
    """ Function that creates
    lyrics: the lyrics we want to predict the genre for
    train_df: Pandas DataFrame with training data
    shots: number of examples to include in the prompt, 0 by default
    """
    # if we don't use 0-shot, add examples to prompt
    examples = []
    if shots > 0:
        examples = ["### EXAMPLES ###\n\n"]
        for _, row in train_df.sample(n=shots, random_state=42).iterrows():
            examples.append(f"""Input: {row.lyrics}\n\nOutput: {row.genre}\n\n""")
    prompt = f"""### SETTING ###\n\nYou are an expert on music, song lyrics, and music genres. Your task is to determine the genre of a song based on its lyrics. For the genres, you can choose only one genre from this list: {genres} Only return the name of the genre. Do not provide any additional information.\n\n{''.join(examples)}### INPUT ###\n\n{lyrics}"""

    # send prompt to model and return a stripped results, because the model sometimes adds newlines before/after the genre
    result = ollama.generate(model="gemma3", prompt=prompt)
    return result["response"].strip()

In [19]:
# read training and test data into Pandas DataFrame
train_df = pd.read_csv("./data/genreLyrics_train.csv", sep="\t")
test_df = pd.read_csv("./data/genreLyrics_test.csv", sep="\t")

# retrieve genres from training data
genres = train_df.genre.unique()

# we'll be using few-shot (2) prompting later on, we're using random_state to always use the same 2 examples for our prompt and print them so we can see which examples were used
print(train_df.sample(n=2, random_state=42))

# prepare lists to store the results
zero_shot_results = []
few_shot_results = []

# iterate through test data
for _, row in test_df.iterrows():
    # prompt model for each lyric and store result (zero-shot)
    zero_shot_results.append(prompt_ollama(row.lyrics, train_df))
    # repeat the same, but use few-shot this time
    few_shot_results.append(prompt_ollama(row.lyrics, train_df, shots=2))

print()

# print classifiction reports to get precision, recall, and f1-scores
print("Zero-shot")
print(classification_report(test_df.genre.to_list(), zero_shot_results, labels=genres))

print("Few-shot")
print(classification_report(test_df.genre.to_list(), few_shot_results, labels=genres))

      Unnamed: 0  genre                                             lyrics
1501       89467  Indie  When I first touched your skin\nThat was when ...
2586      182276   Rock  A black eyed dog he called at my door\nThe bla...

Zero-shot
              precision    recall  f1-score   support

        Rock       0.61      0.35      0.45       655
     Country       0.27      0.53      0.35        89
  Electronic       0.10      0.02      0.03        57
         Pop       0.36      0.46      0.40       255
     Hip-Hop       0.70      0.79      0.74       163
       Metal       0.59      0.46      0.51       151
       Indie       0.03      0.06      0.04        16
         R&B       0.11      0.09      0.10        23
        Folk       0.06      0.53      0.10        17
        Jazz       0.33      0.05      0.08        43
       Other       0.04      0.10      0.05        31

   micro avg       0.41      0.41      0.41      1500
   macro avg       0.29      0.31      0.26      1500
weight

In [20]:
# save results to csv file
combined_results_df = pd.DataFrame({'zero-shot': zero_shot_results, 'few-shot': few_shot_results, 'actual': test_df.genre.to_list()})
combined_results_df.to_csv('results.csv')